In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [2]:
df_stays = pd.read_csv('edstays.csv')

In [3]:
len(df_stays)

425087

In [4]:
df_stays = df_stays.sort_values('intime')

In [5]:
df_stays = df_stays[['subject_id', 'intime', 'outtime', 'stay_id']]

In [6]:
df_triage = pd.read_csv('triage.csv')

In [7]:
df_triage =  df_triage[['subject_id', 'acuity', 'stay_id']]

In [8]:
df = pd.merge(df_stays, df_triage, on=['subject_id', 'stay_id'], how='inner')

In [9]:
len(df)

425087

In [10]:
df.head()

,subject_id,intime,outtime,stay_id,acuity
0,13238787,2110-01-11 01:45:00,2110-01-11 07:04:00,35341790,3.0
1,15350437,2110-01-11 03:43:00,2110-01-11 08:41:00,39042378,2.0
2,13370527,2110-01-11 10:08:00,2110-01-11 19:54:00,35220003,3.0
3,14695442,2110-01-11 12:44:00,2110-01-11 17:59:00,34789981,2.0
4,17195991,2110-01-11 21:42:00,2110-01-12 00:54:00,38649090,NaN


In [11]:
df.subject_id.value_counts()

subject_id
15496609    321
16233333    210
16662316    184
14394983    170
17903094    167
           ... 
11313127      1
19651399      1
12702992      1
10870187      1
14979751      1
Name: count, Length: 205504, dtype: int64

In [12]:
df.head()

,subject_id,intime,outtime,stay_id,acuity
0,13238787,2110-01-11 01:45:00,2110-01-11 07:04:00,35341790,3.0
1,15350437,2110-01-11 03:43:00,2110-01-11 08:41:00,39042378,2.0
2,13370527,2110-01-11 10:08:00,2110-01-11 19:54:00,35220003,3.0
3,14695442,2110-01-11 12:44:00,2110-01-11 17:59:00,34789981,2.0
4,17195991,2110-01-11 21:42:00,2110-01-12 00:54:00,38649090,NaN


In [13]:
df.tail()

,subject_id,intime,outtime,stay_id,acuity
425082,16573705,2212-01-12 12:47:00,2212-01-12 23:51:35,34048700,2.0
425083,11973788,2212-01-19 09:04:00,2212-01-19 15:45:03,33375338,3.0
425084,11936787,2212-01-24 09:59:00,2212-01-24 13:42:00,35476145,3.0
425085,11973788,2212-01-27 20:34:00,2212-01-28 13:17:00,37355521,2.0
425086,11973788,2212-04-05 23:23:00,2212-04-06 14:20:00,33494247,2.0


In [14]:
df = df.dropna()

In [15]:
df

,subject_id,intime,outtime,stay_id,acuity
0,13238787,2110-01-11 01:45:00,2110-01-11 07:04:00,35341790,3.0
1,15350437,2110-01-11 03:43:00,2110-01-11 08:41:00,39042378,2.0
2,13370527,2110-01-11 10:08:00,2110-01-11 19:54:00,35220003,3.0
3,14695442,2110-01-11 12:44:00,2110-01-11 17:59:00,34789981,2.0
5,16077914,2110-01-12 02:01:00,2110-01-12 07:40:43,33855940,2.0
...,...,...,...,...,...
425082,16573705,2212-01-12 12:47:00,2212-01-12 23:51:35,34048700,2.0
425083,11973788,2212-01-19 09:04:00,2212-01-19 15:45:03,33375338,3.0
425084,11936787,2212-01-24 09:59:00,2212-01-24 13:42:00,35476145,3.0
425085,11973788,2212-01-27 20:34:00,2212-01-28 13:17:00,37355521,2.0


In [16]:
df['intime'] = pd.to_datetime(df['intime'])
df['outtime'] = pd.to_datetime(df['outtime'])

In [17]:
df['service_time'] = df['outtime'] - df['intime']
df['service_time_hours'] = df['service_time'].dt.total_seconds() / 3600

In [18]:
mu = (
    df.groupby("acuity")["service_time_hours"]
    .mean()
    .rename("mean_service_time_hrs")
    .to_frame()
)

mu["mu"] = 1 / mu["mean_service_time_hrs"]

In [19]:
mu

,mean_service_time_hrs,mu
acuity,,
1.0,6.091116,0.164174
2.0,8.592253,0.116384
3.0,6.952268,0.143838
4.0,3.491445,0.286414
5.0,2.466184,0.405485


In [20]:
df_sorted = df.sort_values(["acuity", "intime"])
df_sorted["inter_arrival_hrs"] = (
    df_sorted.groupby("acuity")["intime"]
    .diff()
    .dt.total_seconds() / 3600
)

In [21]:
lambda_ = (
    df_sorted.groupby("acuity")["inter_arrival_hrs"]
    .mean()
    .rename("mean_inter_arrival_hrs")
    .to_frame()
)
lambda_["lambda"] = 1 / lambda_["mean_inter_arrival_hrs"]

In [22]:
lambda_

,mean_inter_arrival_hrs,lambda
acuity,,
1.0,36.986441,0.027037
2.0,6.428202,0.155564
3.0,3.974035,0.251633
4.0,31.346251,0.031902
5.0,771.205702,0.001297


In [23]:
rates = lambda_.join(mu)
rates["rho_per_server"] = rates["lambda"] / rates["mu"] 

print(rates[["lambda", "mu", "rho_per_server"]].round(4))

        lambda      mu  rho_per_server
acuity                                
1.0     0.0270  0.1642          0.1647
2.0     0.1556  0.1164          1.3366
3.0     0.2516  0.1438          1.7494
4.0     0.0319  0.2864          0.1114
5.0     0.0013  0.4055          0.0032


In [24]:
# Capacity (K)

In [25]:
events = pd.concat([
    df[["intime"]].assign(change=1).rename(columns={"intime": "time"}),
    df[["outtime"]].assign(change=-1).rename(columns={"outtime": "time"})
]).sort_values("time")

events["occupancy"] = events["change"].cumsum()

In [26]:
occupancy_counts = events["occupancy"].value_counts().sort_index()
print(occupancy_counts)

occupancy
-1          1
 0      15623
 1      53369
 2      95687
 3     125828
 4     135051
 5     125313
 6     102271
 7      74450
 8      49159
 9      29298
 10     15808
 11      7913
 12      3700
 13      1588
 14       662
 15       293
 16       124
 17        41
 18        12
 19         6
 20         3
Name: count, dtype: int64


In [27]:
K_p25 = events["occupancy"].quantile(0.25)

In [28]:
K_p99 = events['occupancy'].quantile(0.99)

In [29]:
K_mean = events['occupancy'].mean()

In [45]:
max(events['occupancy'])

20

In [30]:
K_p25

3.0

In [31]:
K_p99

11.0

In [32]:
K_mean

4.673075819182014

In [34]:
total_mu = sum(rates['mu'])

In [35]:
total_lambda = sum(rates['lambda'])

In [36]:
total_mu

1.1162945966373368

In [37]:
total_lambda

0.46743324070908715

In [42]:
result

4

In [43]:
rho = sum(rates['lambda']/rates['mu']) / result

In [44]:
rho

0.8413345690926695